# Prompt 09 — Frozen TruthfulQA baseline generation
Run on Google Colab T4. Upload the downloaded `AITR_Prompt_09.zip` when prompted. This notebook does **not** generate factual correctness labels or AUROC/AUPRC.

In [ ]:
!pip -q install "transformers==4.57.6" "pandas==2.2.3" "numpy>=2.1,<3" "accelerate==1.15.0" "huggingface-hub>=0.30,<1"

## Upload the Prompt 09 source package
The ZIP contains the verified dataset and frozen sample from your Prompt 08 v4 run.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os, sys
uploaded=files.upload()
packages=[name for name in uploaded if name.endswith(".zip") and "Prompt_09" in name]
assert len(packages)==1, "Upload AITR_Prompt_09.zip"
with zipfile.ZipFile(packages[0]) as z: z.extractall("/content/prompt09_project")
root=Path("/content/prompt09_project")
# Support ZIPs with a single containing folder
folders=list(root.rglob("modules/runner.py"))
assert len(folders)==1, "Expected modules/runner.py in package"
project=folders[0].parent.parent
sys.path.insert(0,str(project/"modules"))
print("Project:",project)

## Optional: restore previous outputs for resume
If you are resuming on a new Colab runtime, upload your prior `prompt09_outputs.zip` now. Otherwise leave this cell unrun.

In [ ]:
from google.colab import files
import zipfile
RESTORE_PREVIOUS = False  # Set True only if you have a prior prompt09_outputs.zip
if RESTORE_PREVIOUS:
    prior=files.upload()
    assert len(prior)==1 and next(iter(prior)).endswith(".zip"), "Upload prior prompt09_outputs.zip"
    with zipfile.ZipFile(next(iter(prior))) as z:
        z.extractall("/content/prompt09_outputs")
    print("Restored previous outputs")
else:
    print("Fresh run; no prior outputs restored")

## Check frozen inputs and unit tests

In [ ]:
from runner import validate_inputs
import unittest
data_path=project/"reference_inputs"/"TruthfulQA_source.csv"
sample_path=project/"reference_inputs"/"frozen_sample_ids.csv"
data,sample=validate_inputs(data_path,sample_path)
print("Frozen dataset:",len(data),"Frozen sample:",len(sample))
loader=unittest.TestLoader().discover(str(project/"modules"),pattern="test_*.py")
result=unittest.TextTestRunner(verbosity=2).run(loader)
assert result.wasSuccessful(), "Unit tests failed"

## Generate and score
Start with five real examples (`RUN_LIMIT=5`), inspect the CSV/JSONL, then set `RUN_LIMIT=None` and rerun to complete 100. This is a **resumable** process: saved positions are skipped. On a new runtime, restore your output ZIP first.

In [ ]:
from runner import run
RUN_LIMIT = 5  # Change to None after checking initial five rows
stats=run(data_path,sample_path,"/content/prompt09_outputs",limit=RUN_LIMIT)
print(stats)

## Inspect real label-free outputs

In [ ]:
import pandas as pd
from pathlib import Path
out=Path("/content/prompt09_outputs")
features=pd.read_csv(out/"features.csv")
display(features.head(10))
print("Completed",len(features),"/ 100")
print("First-10 coverage",features.first_10_mean_nll.notna().sum())
assert not any(x in features.columns for x in ["correctness_label","auroc","auprc"]), "No correctness labels or metrics permitted"

## Download actual Colab output archive
Save this ZIP even after partial runs so you can resume in another session.

In [ ]:
import shutil
from google.colab import files
archive=shutil.make_archive("/content/prompt09_outputs","zip","/content/prompt09_outputs")
files.download(archive)